# Week 5: Advanced business metrics

In [3]:
import duckdb

con = duckdb.connect('../sql/taxi_project')

In [6]:
con.sql(
    """
        SELECT * FROM taxi_clean limit 3;
    """
)

┌─────────────────────┬─────────────────────┬─────────────────┬───────────────┬────────────────┬────────────────┬─────────────┬────────────┐
│   pickup_datetime   │  dropoff_datetime   │ passenger_count │ trip_distance │ pu_location_id │ do_location_id │ fare_amount │ tip_amount │
│      timestamp      │      timestamp      │      int64      │    double     │     int32      │     int32      │   double    │   double   │
├─────────────────────┼─────────────────────┼─────────────────┼───────────────┼────────────────┼────────────────┼─────────────┼────────────┤
│ 2025-01-01 00:18:38 │ 2025-01-01 00:26:59 │               1 │           1.6 │            229 │            237 │        10.0 │        3.0 │
│ 2025-01-01 00:32:40 │ 2025-01-01 00:35:13 │               1 │           0.5 │            236 │            237 │         5.1 │       2.02 │
│ 2025-01-01 00:44:04 │ 2025-01-01 00:46:01 │               1 │           0.6 │            141 │            141 │         5.1 │        2.0 │
└────────────

In [7]:
con.sql(
    """
        SELECT * FROM zone_lookup limit 3;
    """
)

┌────────────┬─────────┬─────────────────────────┬──────────────┐
│ LocationID │ Borough │          Zone           │ service_zone │
│   int64    │ varchar │         varchar         │   varchar    │
├────────────┼─────────┼─────────────────────────┼──────────────┤
│          1 │ EWR     │ Newark Airport          │ EWR          │
│          2 │ Queens  │ Jamaica Bay             │ Boro Zone    │
│          3 │ Bronx   │ Allerton/Pelham Gardens │ Boro Zone    │
└────────────┴─────────┴─────────────────────────┴──────────────┘

What zones have the highest revenue per mile?

In [32]:
con.sql("""
        WITH revenue_per_zone AS (
            SELECT
                t.pu_location_id AS pickup_location,
                z.Zone AS pickup_zone,
                z.Borough AS pickup_borough,
                COUNT(*) AS trips_count,
                SUM(t.trip_distance) AS miles_per_zone,
                SUM(t.fare_amount) AS revenue_per_zone
            FROM taxi_clean AS t
            INNER JOIN zone_lookup AS z
                ON t.pu_location_id = z.LocationID
            WHERE t.fare_amount >= 0
            GROUP BY
                t.pu_location_id,
                z.Zone,
                z.Borough,
            HAVING COUNT(*) >= 100
        ), ranked_revenue_per_zone AS (
            SELECT
                *,
                revenue_per_zone / NULLIF(miles_per_zone, 0) AS revenue_per_mile,
                RANK() OVER (
                    ORDER BY (revenue_per_zone / NULLIF(miles_per_zone, 0)) DESC
                ) AS ranked_revenue_per_zone
            FROM revenue_per_zone
        )
            SELECT
                pickup_zone AS zone,
                pickup_borough AS borough,
                trips_count,
                ROUND(miles_per_zone, 2) AS miles_per_zone,
                ROUND(revenue_per_zone, 2) AS revenue_per_zone,
                ROUND(revenue_per_mile, 2) AS revenue_per_mile, 
                ranked_revenue_per_zone
            FROM ranked_revenue_per_zone
            ORDER BY ranked_revenue_per_zone
            LIMIT 10;
""")

┌──────────────────────────────┬───────────┬─────────────┬────────────────┬──────────────────┬──────────────────┬─────────────────────────┐
│             zone             │  borough  │ trips_count │ miles_per_zone │ revenue_per_zone │ revenue_per_mile │ ranked_revenue_per_zone │
│           varchar            │  varchar  │    int64    │     double     │      double      │      double      │          int64          │
├──────────────────────────────┼───────────┼─────────────┼────────────────┼──────────────────┼──────────────────┼─────────────────────────┤
│ South Ozone Park             │ Queens    │         937 │        5308.59 │         63582.73 │            11.98 │                       1 │
│ Kew Gardens                  │ Queens    │         316 │        1706.12 │         16952.22 │             9.94 │                       2 │
│ Jackson Heights              │ Queens    │         703 │         2899.1 │         27729.88 │             9.56 │                       3 │
│ Woodside          

What routes are long but low revenue?

In [ ]:
con.sql("""
    
""")

What routes are short but highly profitable?

In [ ]:
con.sql("""
    
""")

What hours have the best tip percentage?

In [ ]:
con.sql("""
    
""")

Do airport trips tip better?

In [ ]:
con.sql("""
    
""")

Do longer trips produce higher tip percentage?

In [ ]:
con.sql("""
    
""")

What is the average trip duration by borough?

In [ ]:
con.sql("""
    
""")

Which pickup zones have poor revenue efficiency?

In [ ]:
con.sql("""
    
""")

Which zones have demand but low average fare?

In [ ]:
con.sql("""
    
""")

Which time windows have high demand but low tip rate?

In [ ]:
con.sql("""
    
""")